In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Sonia_Vihar_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,298.0,149.0,192.0,118.0,95.0,104.0,63.0,87.0,147.0,149.0,392.0,365.0
1,2,372.0,204.0,NaN,163.0,104.0,92.0,64.0,96.0,143.0,143.0,375.0,352.0
2,3,397.0,217.0,NaN,136.0,173.0,101.0,111.0,77.0,142.0,152.0,485.0,337.0
3,4,329.0,267.0,119.0,97.0,149.0,160.0,148.0,NaN,136.0,173.0,427.0,307.0
4,5,350.0,254.0,134.0,107.0,241.0,133.0,107.0,94.0,113.0,182.0,454.0,286.0
5,6,385.0,341.0,132.0,139.0,254.0,126.0,62.0,124.0,117.0,218.0,446.0,303.0
6,7,362.0,311.0,163.0,NaN,206.0,219.0,83.0,109.0,108.0,221.0,396.0,343.0
7,8,376.0,128.0,NaN,157.0,151.0,128.0,95.0,126.0,97.0,165.0,444.0,351.0
8,9,425.0,216.0,NaN,210.0,204.0,148.0,NaN,NaN,59.0,183.0,445.0,345.0
9,10,402.0,239.0,NaN,182.0,212.0,133.0,NaN,136.0,58.0,NaN,310.0,321.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    35 non-null     float64
 2   February   33 non-null     float64
 3   March      30 non-null     float64
 4   April      34 non-null     float64
 5   May        37 non-null     float64
 6   June       34 non-null     float64
 7   July       23 non-null     float64
 8   August     27 non-null     float64
 9   September  32 non-null     float64
 10  October    36 non-null     float64
 11  November   34 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Convert all columns except 'Day' to numeric values
for col in df.columns:
    if col != 'Day':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values with column mean
df_filled = df.fillna(df.mean(numeric_only=True))

In [8]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [9]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,298.0,149.0,192.0,118.0,95.0,104.0,63.000000,87.000000,147.0,149.0,392.0,365.0
1,2,372.0,204.0,148.1,163.0,104.0,92.0,64.000000,96.000000,143.0,143.0,375.0,352.0
2,3,397.0,217.0,148.1,136.0,173.0,101.0,78.434783,111.074074,142.0,152.0,485.0,337.0
3,4,329.0,267.0,119.0,97.0,149.0,160.0,78.434783,111.074074,136.0,173.0,427.0,307.0
4,5,350.0,254.0,134.0,107.0,241.0,133.0,78.434783,94.000000,113.0,182.0,454.0,286.0
